In [ ]:
import pandas as pd
import json
import time
import os
from zhipuai import ZhipuAI

API_KEY = "092a2a77831b41fa869790b23dd8efb2.Mg3fnwEQxakb3lRO"  
EXCEL_FILE = "/Users/sylvie/Documents/5508project/副本综合受害者画像_LLM最终版字段xlsx.xlsx"


SAVE_DIR = "/Users/sylvie/Documents/5508project"

MIN_CLUSTERS = 5
MAX_CLUSTERS = 8


os.makedirs(SAVE_DIR, exist_ok=True)


client = ZhipuAI(api_key=API_KEY)

print("正在读取Excel文件...")
df = pd.read_excel(EXCEL_FILE, sheet_name="Sheet1")
df = df.dropna(how='all')

print(f"共读取 {len(df)} 条诈骗案例记录")

cases = []

for idx, row in df.iterrows():
    case = {
        "id": int(idx),
        "情境": str(row.get("情境", "")) if pd.notna(row.get("情境")) else "",
        "LDA_聚类主题": str(row.get("LDA_聚类主题", "")) if pd.notna(row.get("LDA_聚类主题")) else "",
        "受害者年龄": str(row.get("受害者年龄", "")) if pd.notna(row.get("受害者年龄")) else "",
        "受害者性别": str(row.get("受害者性别", "")) if pd.notna(row.get("受害者性别")) else "",
        "受害者职业": str(row.get("受害者职业", "")) if pd.notna(row.get("受害者职业")) else "",
        "受害者经济状况": str(row.get("受害者经济状况", "")) if pd.notna(row.get("受害者经济状况")) else "",
        "关键问题": str(row.get("关键问题", "")) if pd.notna(row.get("关键问题")) else "",
        "情感需求": str(row.get("情感需求", "")) if pd.notna(row.get("情感需求")) else "",
        "信任原因": str(row.get("信任原因", "")) if pd.notna(row.get("信任原因")) else "",
        "心理暗示": str(row.get("心理暗示", "")) if pd.notna(row.get("心理暗示")) else "",
        "动机倾向": str(row.get("动机倾向", "")) if pd.notna(row.get("动机倾向")) else "",
        "压力状态": str(row.get("压力状态", "")) if pd.notna(row.get("压力状态")) else "",
        "诈骗话术": str(row.get("诈骗话术", "")) if pd.notna(row.get("诈骗话术")) else "",
    }
    
    has_content = False
    for v in case.values():
        if isinstance(v, str) and v not in ["", "nan", "None", "无", "NaN"]:
            has_content = True
            break
    if has_content:
        cases.append(case)

print(f"有效案例数: {len(cases)}")

system_prompt = """你是一个诈骗案例分析专家。

【核心要求】：
将受害者群体聚成 5 到 8 类。
聚类时，综合考虑受害者年龄、性别、职业、经济状况、动机倾向、压力状态、信任原因、关键问题等维度。

【输出格式】：
严格按以下JSON格式输出，不要添加额外字段：

{
  "total_clusters": 数字,
  "clusters": [
    {
      "cluster_id": 1,
      "人群名称": "例如：被冒充公检法诈骗的中老年女性",
      
      "受害者特征": {
        "年龄": "从数据归纳，如：60岁以上 / 20-30岁 / 中年 / 不限",
        "性别": "男/女/不限",
        "职业": "从数据归纳",
        "经济状况": "从数据归纳",
        "受害者心理": "从关键问题+情感需求合并提取，如：权威盲从、孤独渴望陪伴、急于赚钱、怕亏怕损失",
        "信任原因": "从数据的'信任原因'字段归纳，如：对方能说出个人信息、出示伪造证件、冒充官方身份等"
      },
      
      "诈骗类型": "从Excel中的'LDA_聚类主题'或'情境'字段中提取判断，冒充身份类可以进一步细分为：冒充公检法、冒充亲属、冒充客服、冒充银行、冒充政府机构、冒充公司高层等；其他类型如：投资理财类、情感婚恋类、求职兼职类等",
      
      "诈骗手段": {
        "诈骗手段": ["从情境、心理暗示等字段综合归纳，可以有多个诈骗手段"],
        "典型话术": [从对应的诈骗案例中的诈骗话术、情境字段中提取，尽量原文，5-6句典型"]
      }
    }
  ]
}

【重要规则】：
1. 只使用我提供的数据，不引入外部诈骗知识
3. 典型话术尽可能从数据中提取原文或贴近原文的概括
4. 诈骗类型从'LDA_聚类主题'或'情境'中综合判断，冒充身份类需进一步细分，例如：
   - 冒充公检法（公安局/检察院/法院）
   - 冒充亲属（子女/孙辈/亲友）
   - 冒充客服（淘宝/支付宝/微信/银行客服）
   - 冒充政府机构（卫生署/水务署/运输署/房委会）
   - 冒充公司高层（CEO/财务总监）
   - 冒充其他（军官/投资专家/老师等）
5. 不要输出：诈骗诉求、损失厌恶心理、情境、综合受害者画像、支撑案例数
"""

user_prompt = f"""请分析以下 {len(cases)} 条诈骗案例数据。

数据字段说明：
- 情境：诈骗发生的具体场景（用于提取诈骗类型和诈骗手段）
- LDA_聚类主题：已有的主题聚类（用于提取诈骗类型，仅供参考）
- 信任原因：为什么会相信骗子
- 动机倾向：贪利/怕亏/情感需求等
- 情感需求：受害者被利用的情感需求
- 关键问题：受害者的心理弱点
- 压力状态：受害者当时的压力情况
- 心理暗示：骗子使用的心理暗示手段
- 受害者年龄、性别、职业、经济状况

数据如下（JSON格式）：
{json.dumps(cases, ensure_ascii=False, indent=2)[:50000]}

【注意】：
1. 受害者特征中的"受害者心理"从"关键问题+情感需求"合并提取
2. 受害者特征中的"信任原因"从数据的"信任原因"字段提取
4. 诈骗类型从LDA_聚类主题或情境中判断，不必完全等于LDA_聚类主题的值
5. 最终聚类数量在5-8类之间

请按指定JSON格式输出。"""


start_time = time.time()

try:
    response = client.chat.completions.create(
        model="glm-4-plus",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        response_format={"type": "json_object"}
    )
    
    elapsed = time.time() - start_time
    print(f"耗时 {elapsed:.1f} 秒")
    
    result = response.choices[0].message.content
    
    try:
        result_json = json.loads(result)
        actual_clusters = len(result_json.get("clusters", []))
        print(f"\n最终聚类数量: {actual_clusters}")
        if actual_clusters < MIN_CLUSTERS or actual_clusters > MAX_CLUSTERS:
            print(f"警告：聚类数量 {actual_clusters} 不在要求的 {MIN_CLUSTERS}-{MAX_CLUSTERS} 范围内")
        
        print("\n各类别概览：")
        for cluster in result_json.get("clusters", []):
            print(f"  {cluster.get('cluster_id')}. {cluster.get('人群名称')}")
            print(f"     诈骗类型: {cluster.get('诈骗类型', '')}")
            
    except Exception as e:
        print(f"解析结果时出错: {e}")
    
    json_path = os.path.join(SAVE_DIR, "victim_cluster_analysis3.json")
    with open(json_path, "w", encoding="utf-8") as f:
        f.write(result)
    print(f"\nJSON结果已保存到: {json_path}")
    
    txt_path = os.path.join(SAVE_DIR, "victim_cluster_analysis3.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("=" * 70 + "\n")
        f.write("诈骗案例受害者群体聚类分析报告\n")
        f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"数据来源: {len(cases)} 条诈骗案例\n")
        f.write("=" * 70 + "\n\n")
        
        try:
            data = json.loads(result)
            f.write(f"总聚类数量: {data.get('total_clusters', len(data.get('clusters', [])))}\n\n")
            
            for cluster in data.get("clusters", []):
                f.write("=" * 60 + "\n")
                f.write(f"【{cluster.get('cluster_id')}】 {cluster.get('人群名称')}\n")
                f.write("=" * 60 + "\n\n")
                
                f.write("受害者特征\n")
                features = cluster.get("受害者特征", {})
                for k, v in features.items():
                    if v and v not in ["", "无"]:
                        f.write(f"  - {k}：{v}\n")
                
                f.write("\n" + "-" * 50 + "\n")
                f.write(f"诈骗类型：{cluster.get('诈骗类型', '')}\n")
                f.write("-" * 50 + "\n\n")
                
                f.write("-" * 50 + "\n")
                f.write("诈骗手段\n")
                f.write("-" * 50 + "\n")
                means = cluster.get("诈骗手段", {})
                
                scam_methods = means.get("诈骗手段", [])
                if scam_methods and isinstance(scam_methods, list):
                    f.write(f"  【典型手段】\n")
                    for method in scam_methods:
                        if method and method not in ["", "无"]:
                            f.write(f"     - {method}\n")
                    f.write("\n")
                
                typical_phrases = means.get("典型话术", [])
                if typical_phrases:
                    f.write(f"  【典型话术】\n")
                    for phrase in typical_phrases:
                        if phrase and phrase not in ["", "无"]:
                            f.write(f"     - \"{phrase}\"\n")
                    f.write("\n")
                
                f.write("-" * 50 + "\n")
            
        except Exception as e:
            f.write(f"解析错误: {e}\n\n")
            f.write("原始返回结果：\n")
            f.write(result)
    
    print(f"TXT结果已保存到: {txt_path}")
    
except Exception as e:
    error_path = os.path.join(SAVE_DIR, "error_log.txt")
    with open(error_path, "w", encoding="utf-8") as f:
        f.write(f"错误: {e}\n")
        f.write(f"时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"\n=== 完成 ===")